# 01b — Nghiệm thu tài nguyên tiền xử lý RPC

**Ngày tạo:** 07/10/2026. **Mục đích:** kiểm tra sâu background, cutout SAM, mask/bbox ảnh ghép và sự sẵn sàng của dữ liệu tiền xử lý. Chỉ đọc `data/archive`; output/manifest/preview được ghi vào `runs/preprocessing_acceptance/<run_id>/`, không sửa dataset và không thực thi scripts đi kèm.

**Phân biệt khái niệm:** background giữ bối cảnh khay và phục vụ copy-paste. GRN là layer của ConvNeXt-V2 giúp tăng cạnh tranh giữa kênh; kiểm tra có background không chứng minh đã giải quyết feature collapse. Nguồn: [paper ConvNeXt-V2](https://arxiv.org/abs/2301.00808).

**Phạm vi:** full pixel decoding và SHA-256 của toàn bộ background không tính preview, cutout được index chấp nhận, 30.000 ảnh checkout và 20.000 ảnh synth; kiểm tra alpha/cutout metadata; polygon–bbox/class/source/visibility của tất cả synth annotations; nguồn background và nhóm train/test. Mask segmentation vẫn là polygon xấp xỉ, không xác nhận khớp pixel owner-map đã mất. Preview chỉ là mẫu QA bằng mắt, không chứng minh toàn bộ cutout đều tách vật đúng.

**Điều kiện nghiệm thu:** tách tài nguyên có thể dùng, nhãn có đúng, split/provenance có đầy đủ và yêu cầu overlap trong handoff có được đáp ứng. Không tự chọn 4/5 task hoặc validation split. Runtime/started_at/finished_at/status được ghi trong notebook.


**Cập nhật sau lần chạy này:** người dùng đã chốt protocol **5 task 100+4×25**. Split/task JSON ảnh thật đã nghiệm thu trong [notebook 02](02_split_and_acceptance.ipynb); xem [báo cáo release](../../docs/data_preprocessing/reports/split-100-4x25-2026-10-07.md). Notebook này giữ output của lần audit/phân tích trước quyết định; nhãn synth được review riêng trong [notebook 03](03_annotation_review.ipynb).


In [1]:
import os,sys,json,uuid,hashlib,io,math,platform,subprocess,time,base64
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import Counter,defaultdict
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image,ImageDraw,ImageFont

TZ=ZoneInfo('Asia/Bangkok')
def now():return datetime.now(TZ).isoformat(timespec='seconds')
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pdp/main.py').exists())
ROOT=Path(os.environ.get('AUTOCHECKOUT_DATA_ROOT',str(REPO/'data/archive'))).resolve()
assert ROOT.is_dir()
run={'run_id':str(uuid.uuid4()),'purpose':'preprocessing resource acceptance, read-only dataset',
     'started_at':now(),'finished_at':None,'timezone':'Asia/Bangkok','python':sys.version.split()[0],
     'host':platform.node(),'git_commit':subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip(),
     'data_root':str(ROOT),'workers':min(8,os.cpu_count() or 1),'status':'RUNNING'}
ARTIFACT=REPO/'runs/preprocessing_acceptance'/run['run_id'];ARTIFACT.mkdir(parents=True,exist_ok=True)
run['artifact_dir']=str(ARTIFACT.relative_to(REPO))
preview_outputs=[]
val=json.loads((ROOT/'instances_val2019.json').read_text())
test=json.loads((ROOT/'instances_test2019.json').read_text())
raw_single=json.loads((ROOT/'instances_train2019.json').read_text())
index=json.loads((ROOT/'cutouts/sam/index.json').read_text())
objects=index['objects'];cut_by_file={o['file']:o for o in objects}
raw_single_anns={a['id']:a for a in raw_single['annotations']}
raw_single_images={i['id']:i for i in raw_single['images']}
real_expected={f'{src}/{i["file_name"]}':(i['width'],i['height']) for src,d in [('val2019',val),('test2019',test)] for i in d['images']}
backgrounds=sorted(p for p in (ROOT/'backgrounds').glob('*.jpg') if p.name!='preview.jpg')
print(json.dumps(run,indent=2));print('Backgrounds:',len(backgrounds),'accepted cutouts:',len(objects))


{
  "run_id": "2e18812d-f053-4fad-83fc-67af1ac0b9b6",
  "purpose": "preprocessing resource acceptance, read-only dataset",
  "started_at": "2026-10-07T11:00:42+07:00",
  "finished_at": null,
  "timezone": "Asia/Bangkok",
  "python": "3.14.4",
  "host": "Laptop",
  "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
  "data_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
  "workers": 8,
  "status": "RUNNING",
  "artifact_dir": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6"
}
Backgrounds: 208 accepted cutouts: 7502


## 1. Full decode ảnh và mask alpha

Đọc pixel thật của tất cả ảnh trong phạm vi, kiểm tra kích thước với annotation/index, SHA-256 file để tìm trùng nội dung nguyên byte. dHash ảnh là công cụ gợi ý QA, không tự kết luận leakage. Nhận diện mask binary ở alpha 0/255 và mask không rỗng. `fill`/IoU SAM được lưu là metadata của pipeline gốc, chưa phải đo lại chất lượng tách vật bằng GT segmentation.


In [2]:
jobs=[]
for p in backgrounds:jobs.append((p.relative_to(ROOT).as_posix(),'background',None))
for o in objects:jobs.append((o['file'],'cutout',tuple(o['size'])))
for rel,size in sorted(real_expected.items()):jobs.append((rel,rel.split('/')[0],size))
for t in range(1,5):
    for p in sorted((ROOT/f'synth/group/task{t}/images').glob('*.jpg')):
        jobs.append((p.relative_to(ROOT).as_posix(),f'synth_task{t}',None))

def inspect_file(job):
    rel,source,expected=job
    record={'file':rel,'source':source,'issues':[]}
    try:
        payload=(ROOT/rel).read_bytes()
        record['sha256']=hashlib.sha256(payload).hexdigest()
        with Image.open(io.BytesIO(payload)) as im:
            im.load();record['size']=list(im.size);record['mode']=im.mode
            if expected and tuple(im.size)!=expected:record['issues'].append('size_mismatch')
            small=np.asarray(im.convert('RGB').resize((64,64)),dtype=np.float32)
            record['pixel_mean']=round(float(small.mean()),3)
            if source=='background':
                record['pixel_std']=round(float(small.std()),3)
            dh=np.asarray(im.convert('L').resize((9,8)),dtype=np.int16)
            record['dhash']=f'{sum(int(v)<<k for k,v in enumerate((dh[:,1:]>dh[:,:-1]).ravel())):016x}'
            if source=='cutout':
                if im.mode!='RGBA':record['issues'].append('not_rgba')
                else:
                    alpha=np.asarray(im.getchannel('A'))
                    binary=(alpha==0)|(alpha==255)
                    if not bool(binary.all()):record['issues'].append('nonbinary_alpha')
                    mask=alpha>127
                    record['alpha_area']=int(mask.sum())
                    if not mask.any():record['issues'].append('empty_mask')
                    if mask.all():record['issues'].append('fully_opaque_mask')
                    record['alpha_fill']=round(float(mask.mean()),6)
    except Exception as error:record['issues'].append(type(error).__name__+': '+str(error))
    return record

file_records=[]
with ThreadPoolExecutor(max_workers=run['workers']) as pool:
    for k,result in enumerate(pool.map(inspect_file,jobs),1):
        file_records.append(result)
        if k%5000==0:print('Decoded',k,'/',len(jobs),flush=True)
image_by_path={r['file']:r for r in file_records}
manifest_path=ARTIFACT/'image_manifest.json'
manifest_path.write_text(json.dumps(file_records,ensure_ascii=False))
image_summary={'files_checked':len(file_records),'by_source':dict(Counter(r['source'] for r in file_records)),
               'files_with_issues':sum(bool(r['issues']) for r in file_records),
               'issues':[(r['file'],r['issues']) for r in file_records if r['issues']],
               'manifest':str(manifest_path.relative_to(REPO)),
               'manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest()}
print(json.dumps(image_summary,ensure_ascii=False,indent=2))


Decoded 5000 / 57710
Decoded 10000 / 57710
Decoded 15000 / 57710
Decoded 20000 / 57710
Decoded 25000 / 57710
Decoded 30000 / 57710
Decoded 35000 / 57710
Decoded 40000 / 57710
Decoded 45000 / 57710
Decoded 50000 / 57710
Decoded 55000 / 57710
{
  "files_checked": 57710,
  "by_source": {
    "background": 208,
    "cutout": 7502,
    "test2019": 24000,
    "val2019": 6000,
    "synth_task1": 5000,
    "synth_task2": 5000,
    "synth_task3": 5000,
    "synth_task4": 5000
  },
  "files_with_issues": 0,
  "issues": [],
  "manifest": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/image_manifest.json",
  "manifest_sha256": "007898b9db9717505c110895f93639dd03c1c13fdc0c016b57e4917d6d4b199a"
}


## 2. Cutout/index và nguồn background

Đối chiếu 200 SKU, source annotation, chất lượng metadata và file ảnh thực tế. Background kiểm tra tên nguồn thuộc val2019/test2019 và kiểm tra preview được loại khỏi danh sách nền. Các tên nguồn không thay thế manifest tham số/run đã dùng để sinh ảnh ghép.


In [3]:
cut_errors=Counter();cut_examples=[]
for o in objects:
    reasons=[]
    a=raw_single_anns.get(o['ann_id'])
    if a is None:reasons.append('unknown_source_annotation')
    else:
        src=raw_single_images.get(a['image_id'],{})
        if a['category_id']!=o['category_id']:reasons.append('source_category_mismatch')
        if src.get('file_name')!=o['source']:reasons.append('source_filename_mismatch')
        if not np.allclose(a['bbox'],o['bbox'],atol=0.01):reasons.append('source_bbox_mismatch')
    params=index['params']
    if not params['min_fill']<=o['fill']<=params['max_fill']:reasons.append('metadata_fill_outside_filter')
    if o['iou']<params['min_iou']:reasons.append('metadata_iou_below_filter')
    if o['file'] not in image_by_path:reasons.append('indexed_file_unchecked')
    for reason in reasons:cut_errors[reason]+=1
    if reasons and len(cut_examples)<10:cut_examples.append({'file':o['file'],'issues':reasons})
coverage=Counter(o['category_id'] for o in objects)
val_names={i['file_name'] for i in val['images']};test_names={i['file_name'] for i in test['images']}
background_summary={'usable_images':len(backgrounds),'preview_excluded':(ROOT/'backgrounds/preview.jpg').exists(),
                    'filename_matches_val2019':sum(p.name in val_names for p in backgrounds),
                    'filename_matches_test2019':sum(p.name in test_names for p in backgrounds),
                    'unknown_source_filenames':[p.name for p in backgrounds if p.name not in val_names|test_names],
                    'sizes':dict(Counter('x'.join(map(str,image_by_path[p.relative_to(ROOT).as_posix()].get('size',[]))) for p in backgrounds)),
                    'source_manifest_exists':any((ROOT/'backgrounds'/n).exists() for n in ['index.json','manifest.json'])}
cutout_summary={'accepted_objects':len(objects),'indexed_unique_files':len(cut_by_file),'classes':len(coverage),
                'missing_rpc_classes':sorted(set(range(1,201))-set(coverage)),
                'minimum_per_class':min(coverage.values()),'maximum_per_class':max(coverage.values()),
                'sam_model':params['sam_model'],'index_params':params,'source_metadata_issues':dict(cut_errors),
                'issue_examples':cut_examples,'source_train2019_present':(ROOT/'train2019').is_dir()}
print('BACKGROUNDS:',json.dumps(background_summary,ensure_ascii=False,indent=2))
print('CUTOUTS:',json.dumps(cutout_summary,ensure_ascii=False,indent=2))


BACKGROUNDS: {
  "usable_images": 208,
  "preview_excluded": true,
  "filename_matches_val2019": 208,
  "filename_matches_test2019": 0,
  "unknown_source_filenames": [],
  "sizes": {
    "1840x1840": 208
  },
  "source_manifest_exists": false
}
CUTOUTS: {
  "accepted_objects": 7502,
  "indexed_unique_files": 7502,
  "classes": 200,
  "missing_rpc_classes": [],
  "minimum_per_class": 10,
  "maximum_per_class": 48,
  "sam_model": "sam2.1_b.pt",
  "index_params": {
    "method": "sam",
    "out": null,
    "per_class": 48,
    "camera_weights": "0:0.5,1:1,2:1.5,3:0.5",
    "classes": null,
    "max_classes": null,
    "pad": 0.1,
    "min_fill": 0.3,
    "max_fill": 0.98,
    "min_iou": 0.9,
    "clip_margin": 0.01,
    "no_clip": false,
    "max_side": 1000,
    "overwrite": false,
    "save_every": 200,
    "seed": 0,
    "sam_model": "sam2.1_b.pt",
    "sam_size": 1024,
    "device": null,
    "scale": 0.5,
    "iters": 5
  },
  "source_metadata_issues": {},
  "issue_examples": [],
  "

## 3. Mask/bbox/visibility và class của mọi ảnh synth

Kiểm tra bbox của polygon theo quy ước pixel inclusive (+1), IoU với bbox annotation >=0.98 như checker nguồn. Kiểm tra diện tích polygon dương và area không lớn hơn bbox; polygon xấp xỉ không bắt buộc có diện tích bằng đúng owner-map. Kiểm tra `cutout` có trong index và SKU khớp task.

Mục tiêu overlap 20–45% trong handoff được đối chiếu với **1 − visible** (phần diện tích vật bị che). Bbox overlap là đại lượng khác. Chưa tự đổi định nghĩa để coi data đạt.


In [4]:
info=json.loads((ROOT/'tasks/group/task_info.json').read_text())
expected_classes={t['task']:set(t['new_classes']) for t in info['tasks']}
synth_summary=[];synth_preview=[]
for t in range(1,5):
    path=ROOT/f'synth/group/task{t}/annotations.json';payload=path.read_bytes();d=json.loads(payload)
    errors=Counter();example_errors=[];counts=Counter();visible=[];area_ratio=[]
    imap={i['id']:i for i in d['images']};by_image=defaultdict(list)
    missing_images=0
    for im in d['images']:
        r=image_by_path.get(im['file_name'])
        if r is None:missing_images+=1
        elif r.get('size')!=[im['width'],im['height']]:errors['actual_image_size_mismatch']+=1
    for a in d['annotations']:
        counts[a['image_id']]+=1;by_image[a['image_id']].append(a)
        reasons=[];image=imap.get(a['image_id']);cid=a['category_id']
        if cid not in expected_classes[t]:reasons.append('category_outside_current_task')
        o=cut_by_file.get(a.get('cutout'))
        if o is None:reasons.append('unknown_cutout_reference')
        elif o['category_id']!=cid:reasons.append('cutout_category_mismatch')
        v=a.get('visible')
        if not isinstance(v,(int,float)) or not 0<v<=1:reasons.append('invalid_visibility')
        else:visible.append(v)
        x,y,w,h=a['bbox'];W,H=image['width'],image['height']
        if w<=0 or h<=0 or x<0 or y<0 or x+w>W+0.01 or y+h>H+0.01:reasons.append('invalid_bbox')
        if not 0<a['area']<=w*h+0.01:reasons.append('invalid_area')
        seg=a.get('segmentation');valid_polys=[]
        if not isinstance(seg,list) or not seg:reasons.append('missing_polygon_mask')
        else:
            for polygon in seg:
                if len(polygon)<6 or len(polygon)%2:reasons.append('invalid_polygon_length');continue
                pts=np.asarray(polygon,dtype=np.float64).reshape(-1,2)
                if not np.isfinite(pts).all():reasons.append('nonfinite_polygon');continue
                if (pts<0).any() or (pts[:,0]>=W).any() or (pts[:,1]>=H).any():reasons.append('polygon_outside_image')
                poly_area=0.5*abs(np.dot(pts[:,0],np.roll(pts[:,1],1))-np.dot(pts[:,1],np.roll(pts[:,0],1)))
                if poly_area<=0:reasons.append('nonpositive_polygon_area')
                valid_polys.append(pts)
            if valid_polys:
                pts=np.concatenate(valid_polys)
                px,py=pts.min(axis=0);pw,ph=pts.max(axis=0)-pts.min(axis=0)+1
                inter=max(0,min(x+w,px+pw)-max(x,px))*max(0,min(y+h,py+ph)-max(y,py))
                iou=inter/(w*h+pw*ph-inter+1e-9)
                if iou<0.98:reasons.append('polygon_bbox_iou_below_0.98')
        for reason in set(reasons):errors[reason]+=1
        if reasons and len(example_errors)<10:example_errors.append({'ann_id':a['id'],'image_id':a['image_id'],'issues':sorted(set(reasons))})
    occlusion=1-np.asarray(visible)
    used={a['category_id'] for a in d['annotations']}
    object_counts=[counts[i['id']] for i in d['images']]
    result={'task':t,'images':len(d['images']),'objects':len(d['annotations']),'classes':len(used),
            'missing_current_classes':sorted(expected_classes[t]-used),'missing_images':missing_images,
            'object_count_min':min(object_counts),'object_count_max':max(object_counts),
            'object_count_mean':round(float(np.mean(object_counts)),3),
            'visible_min':min(visible),'visible_max':max(visible),
            'occlusion_max':round(float(occlusion.max()),3),
            'objects_with_occlusion_20_to_45_percent':int(((occlusion>=0.2-1e-9)&(occlusion<=0.45+1e-9)).sum()),
            'images_with_background_metadata':sum(any(k in im for k in ['background','background_file','background_source']) for im in d['images']),
            'has_generation_config':any(k in d.get('info',{}) for k in ['config','seed','params']),
            'annotation_sha256':hashlib.sha256(payload).hexdigest(),'issues':dict(errors),'issue_examples':example_errors}
    synth_summary.append(result)
    sample_indices=[0,len(d['images'])//2,len(d['images'])-1]
    for k in sample_indices:
        im=d['images'][k];synth_preview.append((t,im,by_image[im['id']]))
    print(json.dumps(result,ensure_ascii=False,indent=2),flush=True)
    del d,payload,by_image


{
  "task": 1,
  "images": 5000,
  "objects": 60473,
  "classes": 48,
  "missing_current_classes": [],
  "missing_images": 0,
  "object_count_min": 4,
  "object_count_max": 20,
  "object_count_mean": 12.095,
  "visible_min": 0.85,
  "visible_max": 1.0,
  "occlusion_max": 0.15,
  "objects_with_occlusion_20_to_45_percent": 0,
  "images_with_background_metadata": 0,
  "has_generation_config": false,
  "annotation_sha256": "447f320003d49e3c6ea8721a39d3694e7554d737ed8ee489c5fd80764f50f7ff",
  "issues": {
    "polygon_bbox_iou_below_0.98": 17
  },
  "issue_examples": [
    {
      "ann_id": 1582,
      "image_id": 137,
      "issues": [
        "polygon_bbox_iou_below_0.98"
      ]
    },
    {
      "ann_id": 10135,
      "image_id": 844,
      "issues": [
        "polygon_bbox_iou_below_0.98"
      ]
    },
    {
      "ann_id": 12682,
      "image_id": 1050,
      "issues": [
        "polygon_bbox_iou_below_0.98"
      ]
    },
    {
      "ann_id": 15643,
      "image_id": 1296,
      "i

## 4. Exact duplicates, nhóm ảnh và QA bằng mắt

So sánh SHA-256 file giữa nguồn train (val2019/synth) và test2019. Kiểm tra suffix nhóm ảnh và SKU multiset dùng bởi `autocheckout/groups.py`. Chỉ xác nhận những tiêu chí đã đo; không coi các quan hệ này là bằng chứng tất cả ảnh gần trùng đã được loại.

Preview: 12 nền, 20 cutout và 12 ảnh synth với polygon/bbox. Người đọc cần kiểm tra nền còn bóng/vật, mask cắt sai và bao bì/chữ bị đảo gương. Code generator nguồn có horizontal flip 50%; archive không lưu transform từng đối tượng để chứng minh mọi ảnh đã dùng chính config đó.


In [5]:
sha_sources=defaultdict(set);sha_examples=defaultdict(list)
for r in file_records:
    if 'sha256' in r:
        sha_sources[r['sha256']].add(r['source']);sha_examples[r['sha256']].append(r['file'])
leak_hashes=[h for h,s in sha_sources.items() if 'test2019' in s and any(x=='val2019' or x.startswith('synth_task') for x in s)]

def multisets(data):
    per=defaultdict(Counter)
    for a in data['annotations']:per[a['image_id']][a['category_id']]+=1
    return {tuple(sorted(per[i['id']].items())) for i in data['images'] if per[i['id']]}
suffix_val={Path(i['file_name']).stem.rsplit('-',1)[-1] for i in val['images']}
suffix_test={Path(i['file_name']).stem.rsplit('-',1)[-1] for i in test['images']}
shared_multisets=multisets(val)&multisets(test)
split_summary={'exact_cross_train_test_hash_matches':len(leak_hashes),
               'hash_examples':[sha_examples[h] for h in leak_hashes[:10]],
               'shared_filename_suffixes':len(suffix_val&suffix_test),
               'shared_nonempty_sku_multisets':len(shared_multisets),
               'independent_validation_manifest_exists':(ROOT/'splits/val.json').exists(),
               'per_image_synth_background_source_available':all(x['images_with_background_metadata']==x['images'] for x in synth_summary),
               'remaining':'Choose protocol, create group-based validation/test manifest, inspect near duplicates and synthetic background provenance'}
print(json.dumps(split_summary,ensure_ascii=False,indent=2))

font=ImageFont.load_default()
def make_sheet(tiles,name,columns=4,tile=(300,330)):
    rows=math.ceil(len(tiles)/columns);sheet=Image.new('RGB',(columns*tile[0],rows*tile[1]),'#ededed')
    for k,(image,label) in enumerate(tiles):
        im=image.copy();im.thumbnail((tile[0]-10,tile[1]-34));x=(k%columns)*tile[0];y=(k//columns)*tile[1]
        sheet.paste(im,(x+(tile[0]-im.width)//2,y+5));ImageDraw.Draw(sheet).text((x+5,y+tile[1]-25),label[:45],fill='black',font=font)
    path=ARTIFACT/name;sheet.save(path)
    buf=io.BytesIO();sheet.save(buf,format='PNG')
    preview_outputs.append({'output_type':'display_data','metadata':{},'data':{'image/png':base64.b64encode(buf.getvalue()).decode(),'text/plain':name}})
    return str(path.relative_to(REPO))
background_tiles=[]
for k in np.linspace(0,len(backgrounds)-1,12,dtype=int):
    p=backgrounds[k]
    with Image.open(p) as im:background_tiles.append((im.convert('RGB'),p.name))
background_preview=make_sheet(background_tiles,'backgrounds.png')
cut_tiles=[]
for k in np.linspace(0,len(objects)-1,20,dtype=int):
    o=objects[k]
    with Image.open(ROOT/o['file']) as im:
        rgba=im.convert('RGBA');rgba.thumbnail((280,280))
        checker=Image.new('RGBA',rgba.size,'#eeeeee');draw=ImageDraw.Draw(checker)
        for y in range(0,rgba.height,16):
            for x in range(0,rgba.width,16):
                if (x//16+y//16)%2:draw.rectangle((x,y,x+15,y+15),fill='#cfcfcf')
        checker.alpha_composite(rgba);cut_tiles.append((checker.convert('RGB'),f"SKU {o['category_id']} | ann {o['ann_id']}"))
cutout_preview=make_sheet(cut_tiles,'cutouts.png')
synth_tiles=[]
for t,imeta,annotations in synth_preview:
    with Image.open(ROOT/imeta['file_name']) as im:
        im=im.convert('RGB');draw=ImageDraw.Draw(im)
        for a in annotations:
            x,y,w,h=a['bbox'];draw.rectangle((x,y,x+w,y+h),outline='red',width=4)
            for poly in a['segmentation']:
                pts=list(zip(poly[::2],poly[1::2]));draw.line(pts+[pts[0]],fill='yellow',width=3)
            draw.text((x,y),str(a['category_id']),fill='red',font=font)
        synth_tiles.append((im,f"Task {t} / image {imeta['id']} / {len(annotations)} objects"))
synth_sheet=make_sheet(synth_tiles,'synth_bbox_masks.png')
print('Previews:',background_preview,cutout_preview,synth_sheet)


{
  "exact_cross_train_test_hash_matches": 0,
  "hash_examples": [],
  "shared_filename_suffixes": 0,
  "shared_nonempty_sku_multisets": 0,
  "independent_validation_manifest_exists": false,
  "per_image_synth_background_source_available": false,
  "remaining": "Choose protocol, create group-based validation/test manifest, inspect near duplicates and synthetic background provenance"
}
Previews: runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/backgrounds.png runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/cutouts.png runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/synth_bbox_masks.png


Preview retained in local notebook snapshot: runs/notebook_versions/01b_preprocessing_acceptance/before-docs-cleanup-20261008T143902+0700.ipynb


Preview retained in local notebook snapshot: runs/notebook_versions/01b_preprocessing_acceptance/before-docs-cleanup-20261008T143902+0700.ipynb


Preview retained in local notebook snapshot: runs/notebook_versions/01b_preprocessing_acceptance/before-docs-cleanup-20261008T143902+0700.ipynb


## 5. Ma trận nghiệm thu và kết thúc

Kết luận từng tầng: tài nguyên background/mask có thể dùng; độ đúng nhãn tổng hợp; split/protocol/provenance/overlap cho run mới. Không kết luận GRN/feature collapse từ kiểm tra data. Notebook kết thúc thành công nghĩa là audit đã chạy, không đồng nghĩa dataset được nghiệm thu toàn bộ.


In [6]:
mask_errors=Counter()
for s in synth_summary:mask_errors.update(s['issues'])
resource_pass=image_summary['files_with_issues']==0 and not cut_errors and len(coverage)==200
annotation_pass=not mask_errors and all(s['missing_images']==0 and not s['missing_current_classes'] for s in synth_summary)
run.update({'finished_at':now(),'status':'AUDIT_COMPLETE'})
run['duration_seconds']=round((datetime.fromisoformat(run['finished_at'])-datetime.fromisoformat(run['started_at'])).total_seconds(),2)
summary={'run':run,'images':image_summary,'backgrounds':background_summary,'cutouts':cutout_summary,
         'synth':synth_summary,'split':split_summary,
         'acceptance':{'resource_integrity':'PASS' if resource_pass else 'FAIL',
                       'synthetic_annotation_geometry':'PASS' if annotation_pass else 'FAIL',
                       'occlusion_20_to_45_percent':(f"FAIL: maximum measured occlusion {max(s['occlusion_max'] for s in synth_summary):.1%}; no object in 20-45% range" if sum(s['objects_with_occlusion_20_to_45_percent'] for s in synth_summary)==0 else 'PARTIAL: occluded examples exist; verify target distribution'),
                       'training_split_protocol':'PENDING: protocol, independent validation and manifest not fixed',
                       'synthetic_provenance':'INCOMPLETE: per-image background/config/transform unavailable',
                       'channel_feature_collapse':'NOT_A_DATA_ACCEPTANCE_CHECK: verify GRN/model features during model stage',
                       'overall':'NOT_FULLY_ACCEPTED_FOR_NEW_EXPERIMENT'},
         'previews':[background_preview,cutout_preview,synth_sheet],
         'limitations':['visual QA is sampled','polygon masks are approximations, original per-image owner masks not stored',
                        'exact SHA matches/suffix/SKU grouping do not exhaust near-duplicate detection',
                        'no 640/800 processor transform or model training was run','no modifications to source data']}
summary_path=ARTIFACT/'summary.json';summary_path.write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('Summary artifact:',str(summary_path.relative_to(REPO)))


{
  "run": {
    "run_id": "2e18812d-f053-4fad-83fc-67af1ac0b9b6",
    "purpose": "preprocessing resource acceptance, read-only dataset",
    "started_at": "2026-10-07T11:00:42+07:00",
    "finished_at": "2026-10-07T11:06:15+07:00",
    "timezone": "Asia/Bangkok",
    "python": "3.14.4",
    "host": "Laptop",
    "git_commit": "6269c0e083f092995e608ec3feed337c0e50b8a5",
    "data_root": "/home/sakana/Code/AutoCheckout-CL/data/archive",
    "workers": 8,
    "status": "AUDIT_COMPLETE",
    "artifact_dir": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6",
    "duration_seconds": 333.0
  },
  "images": {
    "files_checked": 57710,
    "by_source": {
      "background": 208,
      "cutout": 7502,
      "test2019": 24000,
      "val2019": 6000,
      "synth_task1": 5000,
      "synth_task2": 5000,
      "synth_task3": 5000,
      "synth_task4": 5000
    },
    "files_with_issues": 0,
    "issues": [],
    "manifest": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83f

## 6. Chẩn đoán các trường hợp polygon/bbox lệch

Chạy sau audit chính; ghi timestamp riêng. Định lượng mức lệch trên tất cả synth annotations và xuất danh sách lỗi. Không sửa nhãn: bbox được tính từ owner-mask gốc, polygon bị approxPolyDP/min-area-filter nên hai biểu diễn có thể khác nhau. Cần đo mức lệch trước khi kết luận annotation sai.


In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np, json
from pathlib import Path
DIAG_STARTED=datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds')
geometry_issues=[]
for t in range(1,5):
    d=json.loads((ROOT/f'synth/group/task{t}/annotations.json').read_text())
    for a in d['annotations']:
        pts=np.concatenate([np.asarray(p,dtype=float).reshape(-1,2) for p in a['segmentation']])
        px,py=pts.min(axis=0);pw,ph=pts.max(axis=0)-pts.min(axis=0)+1
        x,y,w,h=a['bbox']
        inter=max(0,min(x+w,px+pw)-max(x,px))*max(0,min(y+h,py+ph)-max(y,py))
        iou=inter/(w*h+pw*ph-inter+1e-9)
        if iou<0.98:
            geometry_issues.append({'task':t,'image_id':a['image_id'],'ann_id':a['id'],
                                    'category_id':a['category_id'],'bbox':a['bbox'],
                                    'polygon_bbox':[float(px),float(py),float(pw),float(ph)],
                                    'iou':float(iou),'max_edge_delta_pixels':float(max(abs(x-px),abs(y-py),abs(x+w-px-pw),abs(y+h-py-ph)))})
    del d
geometry_issues.sort(key=lambda x:x['iou'])
issues_path=ARTIFACT/'polygon_bbox_issues.json'
issues_path.write_text(json.dumps(geometry_issues,ensure_ascii=False,indent=2))
ious=np.array([a['iou'] for a in geometry_issues]);deltas=np.array([a['max_edge_delta_pixels'] for a in geometry_issues])
diagnostic={'started_at':DIAG_STARTED,'finished_at':datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds'),
            'issues':len(geometry_issues),'iou_min':float(ious.min()),'iou_median':float(np.median(ious)),
            'iou_max':float(ious.max()),'maximum_edge_delta_pixels':float(deltas.max()),
            'median_edge_delta_pixels':float(np.median(deltas)),'issue_manifest':str(issues_path.relative_to(REPO)),
            'worst_cases':geometry_issues[:5],
            'interpretation':'Polygon simplification/filtering is a plausible source; original owner masks are unavailable, so true segmentation geometry is not certified'}
(ARTIFACT/'geometry_diagnostic.json').write_text(json.dumps(diagnostic,ensure_ascii=False,indent=2))
print(json.dumps(diagnostic,ensure_ascii=False,indent=2))


{
  "started_at": "2026-10-07T11:07:31+07:00",
  "finished_at": "2026-10-07T11:07:37+07:00",
  "issues": 224,
  "iou_min": 0.7266213202222918,
  "iou_median": 0.9446759044153757,
  "iou_max": 0.979999999999886,
  "maximum_edge_delta_pixels": 137.0,
  "median_edge_delta_pixels": 23.0,
  "issue_manifest": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/polygon_bbox_issues.json",
  "worst_cases": [
    {
      "task": 4,
      "image_id": 1240,
      "ann_id": 15271,
      "category_id": 196,
      "bbox": [
        939,
        339,
        254,
        639
      ],
      "polygon_bbox": [
        964.0,
        462.0,
        229.0,
        515.0
      ],
      "iou": 0.7266213202222918,
      "max_edge_delta_pixels": 123.0
    },
    {
      "task": 2,
      "image_id": 659,
      "ann_id": 7318,
      "category_id": 10,
      "bbox": [
        1277,
        0,
        563,
        571
      ],
      "polygon_bbox": [
        1413.0,
        0.0,
        427.0,
    

## 7. QA trực quan và kiểm tra ngưỡng sai số số thực

BBox COCO đỏ, bbox polygon xanh, biên mask vàng. Chỉ tạo preview, không sửa nguồn. Đếm lại ngưỡng IoU với tolerance 1e-9 để loại các trường hợp đúng 0.98 bị báo lỗi do số thực. Một lần thử QA lỗi path relative/absolute đã được ghi trong metadata; cell bên dưới là lần chạy đã sửa.


In [8]:
import json,io,base64
from PIL import Image,ImageDraw,ImageFont
from datetime import datetime
from zoneinfo import ZoneInfo
VIS_STARTED=datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds')
diag=json.loads((ARTIFACT/'geometry_diagnostic.json').read_text())
issues=json.loads((ARTIFACT/'polygon_bbox_issues.json').read_text())
strict_issues=[a for a in issues if a['iou']<0.98-1e-9]
worst=diag['worst_cases'];by_task={}
for t in {a['task'] for a in worst}:
    d=json.loads((ROOT/f'synth/group/task{t}/annotations.json').read_text())
    by_task[t]={a['id']:a for a in d['annotations']}
sheet=Image.new('RGB',(420*len(worst),540),'#eeeeee');font=ImageFont.load_default()
for k,case in enumerate(worst):
    a=by_task[case['task']][case['ann_id']]
    path=ROOT/f"synth/group/task{case['task']}/images/{case['image_id']:06d}.jpg"
    with Image.open(path) as raw:
        image=raw.convert('RGB');draw=ImageDraw.Draw(image)
        x,y,w,h=a['bbox'];px,py,pw,ph=case['polygon_bbox']
        draw.rectangle((x,y,x+w,y+h),outline='red',width=5)
        draw.rectangle((px,py,px+pw,py+ph),outline='#00ff00',width=5)
        for polygon in a['segmentation']:
            pts=list(zip(polygon[::2],polygon[1::2]));draw.line(pts+[pts[0]],fill='yellow',width=3)
        image=image.crop((max(0,x-35),max(0,y-35),min(image.width,x+w+35),min(image.height,y+h+35)))
        image.thumbnail((400,490));sheet.paste(image,(k*420+(420-image.width)//2,5))
    label=f"Task {case['task']} / ann {case['ann_id']} / SKU {case['category_id']} / IoU {case['iou']:.3f}"
    ImageDraw.Draw(sheet).text((k*420+5,515),label,fill='black',font=font)
preview_path=ARTIFACT/'mask_bbox_failures.png';sheet.save(preview_path)
buf=io.BytesIO();sheet.save(buf,format='PNG')
visual_output={'output_type':'display_data','metadata':{},'data':{'image/png':base64.b64encode(buf.getvalue()).decode(),'text/plain':'Worst polygon/bbox discrepancies'}}
visual_run={'started_at':VIS_STARTED,'finished_at':datetime.now(ZoneInfo('Asia/Bangkok')).isoformat(timespec='seconds'),
            'preview':str(preview_path.relative_to(REPO)),'cases':len(worst),'raw_checker_flags':len(issues),
            'strict_flags_with_tolerance':len(strict_issues),'boundary_float_cases':len(issues)-len(strict_issues),
            'iou_below_0_9':sum(x['iou']<0.9 for x in issues)}
(ARTIFACT/'visual_diagnostic.json').write_text(json.dumps(visual_run,indent=2))
print(json.dumps(visual_run,indent=2))


{
  "started_at": "2026-10-07T11:09:49+07:00",
  "finished_at": "2026-10-07T11:09:52+07:00",
  "preview": "runs/preprocessing_acceptance/2e18812d-f053-4fad-83fc-67af1ac0b9b6/mask_bbox_failures.png",
  "cases": 5,
  "raw_checker_flags": 224,
  "strict_flags_with_tolerance": 222,
  "boundary_float_cases": 2,
  "iou_below_0_9": 52
}


Preview retained in local notebook snapshot: runs/notebook_versions/01b_preprocessing_acceptance/before-docs-cleanup-20261008T143902+0700.ipynb
